# 平均・加重平均・ランク平均・blending

最も手軽なアンサンブルである各種平均と blending の使い分けと、重みの決め方を学びます。

> **Kaggle Tutorial** 対応ノートブック（`/learn/ensemble-workflow/02-averaging-blending`）です。上から順にセルを実行（`Shift + Enter` または「ランタイム → すべてのセルを実行」）して動作を確認できます。

In [ ]:
# 3 系統モデルの OOF 予測確率（重み探索用）とテスト予測確率を用意します
import numpy as np
from sklearn.metrics import roc_auc_score

rng = np.random.default_rng(42)
n_train, n_test = 600, 200
y = rng.integers(0, 2, size=n_train)

oof_lgbm = np.clip(0.25 + 0.50 * y + rng.normal(0, 0.22, size=n_train), 0.01, 0.99)
oof_xgb = np.clip(0.27 + 0.46 * y + rng.normal(0, 0.24, size=n_train), 0.01, 0.99)
oof_linear = np.clip(0.30 + 0.40 * y + rng.normal(0, 0.26, size=n_train), 0.01, 0.99)

pred_lgbm = rng.uniform(0.05, 0.95, size=n_test)
pred_xgb = rng.uniform(0.05, 0.95, size=n_test)
pred_linear = rng.uniform(0.05, 0.95, size=n_test)

## 単純平均

最もシンプルなアンサンブルは、複数モデルの予測の**単純平均**です。実装が容易で過学習しにくいのが利点です。

In [ ]:
import numpy as np

pred = (pred_lgbm + pred_xgb + pred_linear) / 3

## 加重平均

モデルの強さに応じて重みを変えます。重みは 交差検証 の OOF 予測で決めると過学習しにくくなります。

In [ ]:
# OOF でスコアが最大になる重みを探索（簡易グリッド）
best = None
for w1 in np.linspace(0, 1, 21):
    for w2 in np.linspace(0, 1 - w1, 21):
        w3 = 1 - w1 - w2
        blend = w1 * oof_lgbm + w2 * oof_xgb + w3 * oof_linear
        score = roc_auc_score(y, blend)
        if best is None or score > best[0]:
            best = (score, (w1, w2, w3))

> **Warning:** 重みは必ず **OOF（検証側）** で最適化します。テストの Public LB を見ながら重みを合わせ込むと、 LB へ過学習してシェイクアップを招きます。

## ランク平均

評価指標が AUC のように**順位**で決まる場合、確率の平均より**順位の平均**が有効なことがあります。
スケールの異なるモデル出力をそろえる効果もあります。

In [ ]:
from scipy.stats import rankdata

rank = (rankdata(pred_lgbm) + rankdata(pred_xgb)) / 2

In [ ]:
print(f"Best OOF AUC: {best[0]:.4f}, optimal weights (lgbm, xgb, linear): {tuple(round(w, 2) for w in best[1])}")
print("Rank-averaged test prediction preview:", rank[:5])

## blending とは

ここまでの「予測値を後段で混ぜる」手法を総称して ブレンディング と呼びます。
学習済みモデルの出力だけを使うため手軽で、コンペ終盤の定番です。

> **Tip:** まずは単純平均から試し、OOF で改善するなら加重・ランク平均へ。指標が順位ベースなら ランク平均、確率の質が重要なら確率平均、と評価指標に合わせて選びます。

## 次のステップ

次は、より強力な stacking（out-of-fold 予測をメタモデルで統合）を学びます。